In [ ]:
#| default_exp compute

# Compute

> EC2 instances, EKS clusters, and ECR container registries.

## Amazon EC2

Equivalent to Azure VM. IMDSv2 enforced, instance profile support.

```python
inst = create_instance(auth, 'my-vm', instance_type='t3.medium')
print(instance_ip(auth, inst['InstanceId']))
```

In [ ]:
#| export
def _ec2(auth):
    return auth.session.client('ec2')

_UBUNTU_FILTER = [
    {'Name': 'name',          'Values': ['ubuntu/images/hvm-ssd/ubuntu-jammy-22.04-amd64-server-*']},
    {'Name': 'architecture',  'Values': ['x86_64']},
    {'Name': 'state',         'Values': ['available']},
    {'Name': 'virtualization-type', 'Values': ['hvm']},
]

def _latest_ubuntu_ami(auth) -> str:
    images = _ec2(auth).describe_images(
        Filters=_UBUNTU_FILTER, Owners=['099720109477'])['Images']
    return sorted(images, key=lambda x: x['CreationDate'], reverse=True)[0]['ImageId']

def create_instance(auth, name, instance_type='t3.medium', ami=None, key_name=None,
                    subnet_id=None, sg_ids=None, iam_instance_profile=None,
                    tags=None, **compliance_opts) -> dict:
    'Launch EC2 instance with IMDSv2 enforced and optional instance profile.'
    ec2 = _ec2(auth)
    image_id = ami or _latest_ubuntu_ami(auth)
    tag_specs = [{'ResourceType': 'instance',
                  'Tags': [{'Key': 'Name', 'Value': name}] +
                          [{'Key': k, 'Value': v} for k, v in (tags or {}).items()]}]
    kwargs = dict(
        ImageId=image_id,
        InstanceType=instance_type,
        MinCount=1, MaxCount=1,
        MetadataOptions={'HttpTokens': 'required',  # IMDSv2
                         'HttpPutResponseHopLimit': 1},
        TagSpecifications=tag_specs,
    )
    if key_name:  kwargs['KeyName'] = key_name
    if subnet_id: kwargs['SubnetId'] = subnet_id
    if sg_ids:    kwargs['SecurityGroupIds'] = sg_ids
    if iam_instance_profile:
        kwargs['IamInstanceProfile'] = {'Name': iam_instance_profile}
    return ec2.run_instances(**kwargs)['Instances'][0]

def instance_ip(auth, instance_id) -> str:
    'Return the public IP address of an EC2 instance.'
    inst = _ec2(auth).describe_instances(
        InstanceIds=[instance_id])['Reservations'][0]['Instances'][0]
    return inst.get('PublicIpAddress', inst.get('PrivateIpAddress', ''))

def start_instance(auth, instance_id):
    'Start a stopped EC2 instance.'
    _ec2(auth).start_instances(InstanceIds=[instance_id])

def stop_instance(auth, instance_id):
    'Stop a running EC2 instance.'
    _ec2(auth).stop_instances(InstanceIds=[instance_id])

def terminate_instance(auth, instance_id):
    'Terminate (permanently delete) an EC2 instance.'
    _ec2(auth).terminate_instances(InstanceIds=[instance_id])

## Amazon EKS

Equivalent to Azure AKS. Managed Kubernetes with node group.

```python
create_eks(auth, 'my-cluster', node_count=3)
print(eks_kubeconfig(auth, 'my-cluster'))
```

In [ ]:
#| export
import subprocess

def _eks(auth):
    return auth.session.client('eks')

def create_eks(auth, name, node_type='m5.large', node_count=3,
               version='1.31', subnet_ids=None, sg_ids=None,
               tags=None, **compliance_opts) -> dict:
    'Create EKS cluster with managed node group.'
    from .network import create_role, attach_policy
    client = _eks(auth)

    # Cluster IAM role
    cluster_role = create_role(
        auth, f'{name}-eks-cluster-role',
        trust_policy=_eks_cluster_trust())
    attach_policy(auth, f'{name}-eks-cluster-role',
                  'arn:aws:iam::aws:policy/AmazonEKSClusterPolicy')

    # Node group IAM role
    node_role = create_role(
        auth, f'{name}-eks-node-role',
        trust_policy=_ec2_trust())
    for policy in ['AmazonEKSWorkerNodePolicy',
                   'AmazonEKS_CNI_Policy',
                   'AmazonEC2ContainerRegistryReadOnly']:
        attach_policy(auth, f'{name}-eks-node-role',
                      f'arn:aws:iam::aws:policy/{policy}')

    resources_vpc = {}
    if subnet_ids: resources_vpc['subnetIds'] = subnet_ids
    if sg_ids:     resources_vpc['securityGroupIds'] = sg_ids

    try:
        cluster = client.create_cluster(
            name=name,
            version=version,
            roleArn=cluster_role['Role']['Arn'],
            resourcesVpcConfig=resources_vpc,
            tags=tags or {},
        )['cluster']
    except client.exceptions.ResourceInUseException:
        cluster = client.describe_cluster(name=name)['cluster']

    # Managed node group
    try:
        client.create_nodegroup(
            clusterName=name,
            nodegroupName=f'{name}-ng',
            scalingConfig={'minSize': 1, 'maxSize': node_count * 2,
                           'desiredSize': node_count},
            instanceTypes=[node_type],
            nodeRole=node_role['Role']['Arn'],
            subnets=subnet_ids or [],
        )
    except client.exceptions.ResourceInUseException:
        pass

    return cluster

def eks_kubeconfig(auth, name) -> str:
    'Generate and return kubeconfig for an EKS cluster (requires aws CLI).'
    result = subprocess.run(
        ['aws', 'eks', 'update-kubeconfig', '--name', name,
         '--region', auth.region, '--dry-run'],
        capture_output=True, text=True)
    return result.stdout

def scale_eks(auth, name, node_count):
    'Update desired node count on the default node group.'
    _eks(auth).update_nodegroup_config(
        clusterName=name,
        nodegroupName=f'{name}-ng',
        scalingConfig={'desiredSize': node_count})

def _eks_cluster_trust() -> dict:
    return {'Version': '2012-10-17', 'Statement': [{
        'Effect': 'Allow',
        'Principal': {'Service': 'eks.amazonaws.com'},
        'Action': 'sts:AssumeRole'}]}

def _ec2_trust() -> dict:
    return {'Version': '2012-10-17', 'Statement': [{
        'Effect': 'Allow',
        'Principal': {'Service': 'ec2.amazonaws.com'},
        'Action': 'sts:AssumeRole'}]}

## Amazon ECR

Equivalent to Azure Container Registry. Scan-on-push enabled by default.

```python
create_ecr(auth, 'my-app')
print(ecr_login_url(auth, 'my-app'))
```

In [ ]:
#| export
def _ecr(auth):
    return auth.session.client('ecr')

def create_ecr(auth, name, scan_on_push=True, tags=None) -> dict:
    'Create ECR repository with scan-on-push enabled.'
    client = _ecr(auth)
    tag_list = [{'Key': k, 'Value': v} for k, v in (tags or {}).items()]
    try:
        return client.create_repository(
            repositoryName=name,
            imageScanningConfiguration={'scanOnPush': scan_on_push},
            encryptionConfiguration={'encryptionType': 'AES256'},
            tags=tag_list,
        )['repository']
    except client.exceptions.RepositoryAlreadyExistsException:
        return client.describe_repositories(
            repositoryNames=[name])['repositories'][0]

def ecr_login_url(auth, name) -> str:
    'Return the ECR repository URI (host/name for docker push/pull).'
    return f'{auth.account_id}.dkr.ecr.{auth.region}.amazonaws.com/{name}'

def attach_ecr_to_eks(auth, ecr_name, eks_name):
    'Grant EKS node role AmazonEC2ContainerRegistryReadOnly access to ECR.'
    from .network import attach_policy
    attach_policy(auth, f'{eks_name}-eks-node-role',
                  'arn:aws:iam::aws:policy/AmazonEC2ContainerRegistryReadOnly')